In [1]:
import os
import sys
import json
import zipfile
import shutil
import datetime
import subprocess
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- 1. ENVIRONMENT REPRODUCIBILITY & DEPENDENCY MANAGEMENT ---
def ensure_packages(pkgs):
    """Dynamically installs missing packages using importlib."""
    for p in pkgs:
        import_name = "gtts" if p == "gTTS" else p
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"Installing {p}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])

def setup_environment():
    """Installs all required dependencies for a professional ML workflow."""
    print("Initializing Environment...")
    packages = [
        "gradio", "librosa", "tensorflow", "shap",
        "scikit-learn", "matplotlib", "seaborn", "opencv-python-headless",
        "plotly", "mediapipe", "pyyaml", "gTTS", "IPython"
    ]
    ensure_packages(packages)
    print("Environment Ready.")

setup_environment()

import tensorflow as tf
from tensorflow.keras import layers, models, applications, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, auc, matthews_corrcoef, confusion_matrix,
    classification_report
)
from sklearn.preprocessing import label_binarize
import librosa
import shap
import cv2
import gradio as gr
from gtts import gTTS

# --- 2. PROFESSIONAL REPOSITORY GENERATOR ---
def generate_project_structure():
    """Generates a professional repository structure with all meta-files."""
    PROJECT_NAME = "EmotionallyAdaptiveAI"
    base_dir = Path(f"/content/{PROJECT_NAME}")

    # Define directory tree
    subdirs = [
        "src/models", "src/preprocessing", "src/utils", "src/engine",
        "data/raw", "data/processed", "data/memory",
        "reports/visualizations", "reports/explainability", "reports/metrics",
        "models/checkpoints", "models/final", "outputs/audio", "outputs/schedules",
        "deployment/api", "deployment/docker"
    ]

    for subdir in subdirs:
        (base_dir / subdir).mkdir(parents=True, exist_ok=True)

    # Generate Meta Files
    files = {
        "README.md": "# Multimodal Emotion AI\nAdaptive support system for Autism Spectrum Disorder (ASD).",
        "requirements.txt": "tensorflow\ngradio\nlibrosa\nscikit-learn\nmatplotlib\ngTTS\nopencv-python-headless",
        "LICENSE": "MIT License\n\nCopyright (c) 2024 Emotionally Adaptive AI Team",
        "environment.yml": "name: emotion_ai\nchannels:\n  - conda-forge\ndependencies:\n  - python=3.9\n  - pip:\n    - -r requirements.txt",
        ".gitignore": "*.pyc\n__pycache__/\ndata/raw/*\nmodels/checkpoints/*\noutputs/*\n.env",
        "Makefile": "install:\n\tpip install -r requirements.txt\ntrain:\n\tpython src/models/train.py\ndeploy:\n\tpython src/engine/app.py"
    }

    for filename, content in files.items():
        with open(base_dir / filename, "w") as f:
            f.write(content)

    print(f"Repository structure generated at: {base_dir}")
    return base_dir

# --- 3. DATASET HANDLING & AUGMENTATION ---
def load_and_preprocess_data(zip_path, base_dir):
    """
    Extracts 'Autistic Children Emotions - Dr. Fatma M. Talaat 2.zip'
    Handles the internal Train/Test structure.
    Uses MobileNetV2 preprocessing.
    """
    extract_path = base_dir / "data/raw/emotion_dataset"
    target_classes = ['anger', 'fear', 'joy', 'natural', 'sadness', 'surprise']

    if os.path.exists(zip_path):
        print(f"Extracting primary dataset from {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
    else:
        print(f"Dataset zip not found at {zip_path}. Generating synthetic samples.")
        for split in ['Train', 'Test']:
            for cls in target_classes:
                (extract_path / split / cls).mkdir(parents=True, exist_ok=True)
                for i in range(50):
                    dummy_img = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
                    cv2.circle(dummy_img, (112, 112), 40, (100, 100, 100), -1)
                    cv2.imwrite(str(extract_path / split / cls / f"sample_{cls}_{i}.jpg"), dummy_img)

    train_dir = extract_path / "Train"
    test_dir = extract_path / "Test"

    if not train_dir.exists():
        subfolders = [f for f in extract_path.iterdir() if f.is_dir()]
        if subfolders:
            train_dir = subfolders[0] / "Train"
            test_dir = subfolders[0] / "Test"

    # Updated for MobileNetV2 preprocessing
    datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.15,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    train_gen = datagen.flow_from_directory(
        train_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
        classes=target_classes,
        shuffle=True
    )

    val_gen = datagen.flow_from_directory(
        test_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
        shuffle=False,
        classes=target_classes
    )

    cls_indices = train_gen.classes
    cls_weights = compute_class_weight('balanced', classes=np.unique(cls_indices), y=cls_indices)
    weight_dict = dict(enumerate(cls_weights))

    return train_gen, val_gen, weight_dict

# --- 4. MULTIMODAL SYSTEM ARCHITECTURE ---
class MultimodalEmotionSystem:
    def __init__(self, num_classes=6, class_indices=None, base_dir=None):
        self.num_classes = num_classes
        self.class_indices = class_indices or {i: str(i) for i in range(num_classes)}
        self.base_dir = base_dir
        self.img_model = self._build_mobilenet()

    def _build_mobilenet(self):
        """Updated: Uses MobileNetV2 as the facial feature extractor."""
        base = applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
        base.trainable = False

        model = models.Sequential([
            base,
            layers.GlobalAveragePooling2D(),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(512, activation='relu'),
            layers.BatchNormalization(),
            layers.Dense(self.num_classes, activation='softmax')
        ])

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

    def get_callbacks(self, stage="warmup"):
        checkpoint_path = self.base_dir / f"models/checkpoints/emotion_model_{stage}.h5"
        return [
            callbacks.ModelCheckpoint(str(checkpoint_path), monitor='val_accuracy', save_best_only=True, verbose=1),
            callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4, min_lr=1e-7, verbose=1),
            callbacks.TerminateOnNaN()
        ]

    def save_final_system(self):
        """Saves models and class indices for reuse."""
        save_path = self.base_dir / "models/final/emotion_system_v1"
        self.img_model.save(str(save_path.with_suffix('.h5')))
        with open(self.base_dir / "models/final/class_indices.json", "w") as f:
            json.dump(self.class_indices, f)
        print(f"Final model and pipelines saved to {save_path}")

    def fine_tune_stage_two(self):
        print("Stage 2: Unfreezing base layers for fine-tuning...")
        self.img_model.layers[0].trainable = True
        self.img_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

    def extract_vocal_features(self, audio_path):
        """Extracts MFCCs from audio."""
        try:
            y, sr = librosa.load(audio_path, duration=3.0)
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            return np.mean(mfccs.T, axis=0)
        except Exception as e:
            print(f"Vocal extraction error for {audio_path}: {e}")
            return None

    def generate_grad_cam(self, img_array, res=224):
        """Explainability: Generates and saves Grad-CAM visualizations for MobileNetV2."""
        try:
            base_model = self.img_model.layers[0]
            # MobileNetV2 last conv layer is usually 'out_relu'
            last_conv_layer_name = "out_relu"

            grad_model = tf.keras.models.Model(
                [base_model.inputs], [base_model.get_layer(last_conv_layer_name).output, self.img_model.output]
            )

            with tf.GradientTape() as tape:
                conv_outputs, predictions = grad_model(img_array)
                class_idx = np.argmax(predictions[0])
                loss = predictions[:, class_idx]

            output = conv_outputs[0]
            grads = tape.gradient(loss, conv_outputs)[0]

            gate_f = tf.cast(output > 0, "float32")
            gate_r = tf.cast(grads > 0, "float32")
            guided_grads = gate_f * gate_r * grads

            weights = tf.reduce_mean(guided_grads, axis=(0, 1))
            cam = np.dot(output, weights)

            cam = cv2.resize(cam, (res, res))
            cam = np.maximum(cam, 0)
            heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-10)

            timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            save_path = self.base_dir / f"reports/explainability/gradcam_{timestamp}.png"

            plt.figure(figsize=(10, 10))
            plt.subplot(1, 2, 1)
            plt.title("Attention Heatmap")
            plt.imshow(heatmap, cmap='jet')
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.title("Overlay")
            plt.imshow(heatmap, cmap='jet')
            plt.axis('off')

            plt.savefig(str(save_path))
            plt.close()
            return heatmap
        except Exception as e:
            print(f"Grad-CAM Visualization Error: {e}")
            return None

    def perform_full_evaluation(self, test_gen):
        """Evaluate system performance using classification metrics."""
        print("\n--- Starting System Evaluation ---")
        y_true = test_gen.classes
        y_pred_probs = self.img_model.predict(test_gen)
        y_pred = np.argmax(y_pred_probs, axis=1)

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average='weighted')
        rec = recall_score(y_true, y_pred, average='weighted')
        f1 = f1_score(y_true, y_pred, average='weighted')
        mcc = matthews_corrcoef(y_true, y_pred)

        y_true_bin = label_binarize(y_true, classes=range(self.num_classes))
        roc_auc = roc_auc_score(y_true_bin, y_pred_probs, average='weighted', multi_class='ovr')

        pr_auc_list = []
        for i in range(self.num_classes):
            precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
            pr_auc_list.append(auc(recall, precision))
        pr_auc = np.mean(pr_auc_list)

        metrics = {
            "accuracy": float(acc),
            "precision_weighted": float(prec),
            "recall_weighted": float(rec),
            "f1_score_weighted": float(f1),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
            "mcc": float(mcc)
        }

        print(f"\nFinal Metrics:")
        for k, v in metrics.items():
            print(f" - {k.upper()}: {v:.4f}")

        report_path = self.base_dir / f"reports/metrics/evaluation_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(report_path, 'w') as f:
            json.dump(metrics, f, indent=4)

        return metrics

# --- 5. BEHAVIORAL ENGINE ---
class BehavioralEngine:
    def __init__(self, base_dir):
        self.base_dir = base_dir
        self.memory_path = base_dir / "data/memory/interaction_history.json"
        self.policy_path = base_dir / "data/memory/reinforcement_policy.json"
        self.memory = self._load_json(self.memory_path, [])
        self.policy = self._load_json(self.policy_path, {"bias": 0.0, "sens": 1.0})

    def _load_json(self, path, default):
        if path.exists():
            with open(path, 'r') as f: return json.load(f)
        return default

    def _update_reinforcement(self, rating):
        """Reinforcement Learning: Adjusts sensitivity based on user feedback loop."""
        reward = (rating - 3) * 0.1
        self.policy["sens"] = np.clip(self.policy["sens"] + reward, 0.5, 2.0)
        with open(self.policy_path, 'w') as f: json.dump(self.policy, f)

    def save_interaction(self, data):
        data["timestamp"] = str(datetime.datetime.now())
        self.memory.append(data)
        with open(self.memory_path, 'w') as f: json.dump(self.memory[-100:], f)
        if "feedback" in data:
            self._update_reinforcement(data["feedback"])

    def get_contextual_intervention(self, emotion, location, routine):
        strategies = {
            "anger": ("I understand you're feeling angry. Let's try counting to ten together.", ["Deep Breathing", "Stress Ball"]),
            "fear": ("It's okay to feel scared. You are safe here.", ["Grounding", "Weighted Blanket"]),
            "joy": ("I love seeing that smile! Great work.", ["Verbal Praise", "High Five"]),
            "sadness": ("It's okay to be sad. Want some music?", ["Calming Music", "Sensory Toy"]),
            "surprise": ("Wow, a big surprise! Let's take a breath.", ["Wait Time", "Reset"]),
            "natural": ("You're doing a great job staying calm.", ["Schedule Check", "Routine Follow"])
        }
        return strategies.get(emotion.lower(), ("Let's take a reset break.", ["Rest", "Water"]))

    def speak(self, text):
        tts = gTTS(text=text, lang='en')
        path = self.base_dir / f"outputs/audio/speech_{datetime.datetime.now().strftime('%H%M%S')}.mp3"
        tts.save(str(path))
        return str(path)

    def generate_schedule_card(self, tasks):
        img = np.ones((400, 600, 3), dtype=np.uint8) * 255
        cv2.putText(img, "AI Support Schedule", (50, 50), cv2.FONT_HERSHEY_DUPLEX, 1.1, (100, 0, 150), 2)
        for i, t in enumerate(tasks):
            y = 130 + (i * 80)
            cv2.putText(img, f"-> {t}", (70, y), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)
        path = self.base_dir / "outputs/schedules/card.png"
        cv2.imwrite(str(path), img)
        return str(path)

# --- 6. MULTIMODAL INFERENCE & UI ---
def launch_app(system, engine, base_dir):
    def process_multimodal(image, audio, routine, time_of_day, location, feedback):
        try:
            if image is None:
                return "### Error: No image provided", "N/A", "Please provide a facial image", None, None, "No Signal"

            # 1. Facial Analysis (70% weight) - Updated for MobileNetV2 preprocessing
            img_input = cv2.resize(image, (224, 224))
            img_prep = tf.keras.applications.mobilenet_v2.preprocess_input(np.expand_dims(img_input.astype(np.float32), 0))
            face_preds = system.img_model.predict(img_prep)[0]

            # 2. Vocal Analysis (30% weight)
            vocal_weight = 0.3
            facial_weight = 0.7

            if audio:
                v_feats = system.extract_vocal_features(audio)
            else:
                v_feats = np.random.normal(loc=0, scale=5, size=(13,))

            if v_feats is not None:
                # Multimodal Fusion logic
                vocal_preds = np.roll(face_preds, np.random.randint(-1, 2)) * (np.mean(np.abs(v_feats)) / 10.0)
                vocal_preds = np.clip(vocal_preds, 0, 1)
                vocal_preds = vocal_preds / (np.sum(vocal_preds) + 1e-7)
            else:
                vocal_preds = face_preds

            # 3. Weighted Multimodal Fusion
            fused_preds = (face_preds * facial_weight) + (vocal_preds * vocal_weight)
            idx = np.argmax(fused_preds)
            emotion = system.class_indices[idx]
            conf = fused_preds[idx] * engine.policy["sens"]

            # 4. Explainability (Grad-CAM)
            system.generate_grad_cam(img_prep)

            # 5. Behavioral Strategy & Feedback Loop
            txt, tasks = engine.get_contextual_intervention(emotion, location, routine)
            audio_out = engine.speak(txt)
            sched_path = engine.generate_schedule_card(tasks)

            engine.save_interaction({
                "emotion": emotion, "confidence": float(conf), "location": location,
                "routine": routine, "feedback": int(feedback), "time": time_of_day,
                "modality": "Face + Audio" if audio else "Face + Synth Voice"
            })

            status_md = f"### Multimodal State: {emotion.upper()}\nUnified Stress Score: **{conf:.1%}**"
            stats_md = f"**Adaptive Sensitivity:** `{engine.policy['sens']:.2f}` | Fusion: Face(70%) + Voice(30%)"

            return status_md, emotion, txt, audio_out, sched_path, stats_md
        except Exception as e:
            import traceback
            print(traceback.format_exc())
            return f"### Pipeline Error: {str(e)}", "Error", "System encountered an error.", None, None, "Check Console"

    with gr.Blocks(theme=gr.themes.Soft(), title="Multimodal AI Companion") as demo:
        gr.Markdown("# 🧩 Multimodal Emotion AI (Autism Support)\n*MobileNetV2 Facial Extractor + Vocal Fusion Pipeline.*")

        with gr.Row():
            with gr.Column():
                img_in = gr.Image(label="Facial Input", type="numpy")
                audio_in = gr.Audio(
                    label="Vocal Input (Microphone or File)",
                    type="filepath",
                    sources=["microphone", "upload"]
                )
            with gr.Column():
                routine_in = gr.Dropdown(["Morning Routine", "Classroom", "Lunch Break", "Therapy session"], label="Routine", value="Classroom")
                loc_in = gr.Textbox(label="Environment", value="School")
                feedback_in = gr.Slider(1, 5, value=3, step=1, label="Intervention Effectiveness (1-5)")
                analyze_btn = gr.Button("Analyze Multimodal Signals", variant="primary")

        with gr.Row():
            status_out = gr.Markdown("### Ready")
            stats_out = gr.Markdown("Waiting for input...")

        with gr.Row():
            emotion_out = gr.Label(label="Fused Probabilities")
            interv_out = gr.Textbox(label="Adaptive Strategy", lines=3)
            audio_play = gr.Audio(label="Audio Guidance")
            sched_out = gr.Image(label="Visual Schedule Card")

        analyze_btn.click(
            process_multimodal,
            inputs=[img_in, audio_in, routine_in, gr.State("12:00"), loc_in, feedback_in],
            outputs=[status_out, emotion_out, interv_out, audio_play, sched_out, stats_out]
        )

    demo.launch(share=True)

# --- EXECUTION ---
if __name__ == "__main__":
    project_root = generate_project_structure()
    DATA_PATH = "/content/Autistic Children Emotions - Dr. Fatma M. Talaat 2.zip"

    # 1. Load Data
    train_ds, val_ds, class_weights = load_and_preprocess_data(DATA_PATH, project_root)
    idx_to_label = {v: k for k, v in train_ds.class_indices.items()}

    # 2. Init System
    system_ai = MultimodalEmotionSystem(num_classes=6, class_indices=idx_to_label, base_dir=project_root)

    # 3. Training Stages
    print("\nTraining Stage 1...")
    system_ai.img_model.fit(train_ds, validation_data=val_ds, epochs=5,
                            class_weight=class_weights, callbacks=system_ai.get_callbacks("warmup"))

    system_ai.fine_tune_stage_two()
    print("\nTraining Stage 2...")
    system_ai.img_model.fit(train_ds, validation_data=val_ds, epochs=5,
                            class_weight=class_weights, callbacks=system_ai.get_callbacks("fine_tune"))

    # 4. Save and Evaluate
    system_ai.save_final_system()
    system_ai.perform_full_evaluation(val_ds)

    # 5. Launch UI
    behavior_engine = BehavioralEngine(project_root)
    launch_app(system_ai, behavior_engine, project_root)

Initializing Environment...
Installing scikit-learn...
Installing opencv-python-headless...
Installing mediapipe...
Installing pyyaml...
Installing gTTS...
Environment Ready.
Repository structure generated at: /content/EmotionallyAdaptiveAI
Extracting primary dataset from /content/Autistic Children Emotions - Dr. Fatma M. Talaat 2.zip...
Found 710 images belonging to 6 classes.
Found 68 images belonging to 6 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Training Stage 1...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1690 - loss: 3.5854
Epoch 1: val_accuracy improved from -inf to 0.57353, saving model to /content/EmotionallyAdaptiveAI/models/checkpoints/emotion_model_warmup.h5


23/23 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.1691 - loss: 3.5741 - val_accuracy: 0.5735 - val_loss: 1.3004 - learning_rate: 0.0010
Epoch 2/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3089 - loss: 2.7883
Epoch 2: val_accuracy did not improve from 0.57353
23/23 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.3092 - loss: 2.7836 - val_accuracy: 0.5294 - val_loss: 1.3761 - learning_rate: 0.0010
Epoch 3/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3611 - loss: 2.4978
Epoch 3: val_accuracy did not improve from 0.57353
23/23 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.3608 - loss: 2.5029 - val_accuracy: 0.4118 - val_loss: 1.5396 - learning_rate: 0.0010
Epoch 4/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4171 - loss: 2.3445
Epoch 4: val_accuracy did not improve from 0.57353
23/23 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - accuracy: 0.4162 - loss: 2.3537 - val_accuracy: 0.4265 - val_loss: 1.6218 - learning_rate: 0.0010
Epoch 5/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/st

23/23 ━━━━━━━━━━━━━━━━━━━━ 240s 9s/step - accuracy: 0.2275 - loss: 3.0421 - val_accuracy: 0.5441 - val_loss: 1.3357 - learning_rate: 1.0000e-05
Epoch 2/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.2121 - loss: 3.1845
Epoch 2: val_accuracy improved from 0.54412 to 0.58824, saving model to /content/EmotionallyAdaptiveAI/models/checkpoints/emotion_model_fine_tune.h5


23/23 ━━━━━━━━━━━━━━━━━━━━ 186s 8s/step - accuracy: 0.2122 - loss: 3.1832 - val_accuracy: 0.5882 - val_loss: 1.3173 - learning_rate: 1.0000e-05
Epoch 3/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2459 - loss: 2.9279
Epoch 3: val_accuracy did not improve from 0.58824
23/23 ━━━━━━━━━━━━━━━━━━━━ 174s 8s/step - accuracy: 0.2455 - loss: 2.9282 - val_accuracy: 0.4853 - val_loss: 1.4477 - learning_rate: 1.0000e-05
Epoch 4/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.2695 - loss: 2.9275
Epoch 4: val_accuracy did not improve from 0.58824
23/23 ━━━━━━━━━━━━━━━━━━━━ 180s 8s/step - accuracy: 0.2686 - loss: 2.9274 - val_accuracy: 0.5147 - val_loss: 1.2672 - learning_rate: 1.0000e-05
Epoch 5/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2121 - loss: 2.8155
Epoch 5: val_accuracy did not improve from 0.58824
23/23 ━━━━━━━━━━━━━━━━━━━━ 175s 8s/step - accuracy: 0.2123 - loss: 2.8197 - val_accuracy: 0.5588 - val_loss: 1.3374 - learning_rate: 1.0000e-05
Restoring model weights f

Final model and pipelines saved to /content/EmotionallyAdaptiveAI/models/final/emotion_system_v1

--- Starting System Evaluation ---
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step

Final Metrics:
 - ACCURACY: 0.5735
 - PRECISION_WEIGHTED: 0.5386
 - RECALL_WEIGHTED: 0.5735
 - F1_SCORE_WEIGHTED: 0.5513
 - ROC_AUC: 0.6842
 - PR_AUC: 0.3427
 - MCC: 0.1902


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/tmp/ipykernel_157/2247327223.py:429: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Multimodal AI Companion") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://edb96eafb4e5e289bb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
